# Summary

In this notebook, we create (1) a Biguery table with IPEDS data and (2) a text document with a data dictionary and other content explaining the text.

In [1]:
import os, sys
import pandas as pd
import pandas_gbq as pbq

from google.cloud import bigquery

utils_path = "../../../../ccc-policy_assistant/interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication
import gcp_tools as gct

# Set environment variables
dotenv_path = "../../../../ccc-policy_assistant/data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()


## Read file from Cloud storage

In [2]:
gcs_bucket_name = "ccc-crawl_data"
gcs_directory = "ipeds/zipcsv_files/prep"

# Get a list of files in the GCP bucket
# gct.gcp_list_bucket(gcp_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
#                     gcs_bucket_name=gcs_bucket_name,
#                     path=gcs_directory)


In [3]:
# Get the dataframe with table descriptions from the GCP
df_desc = gct.read_csv_file_into_pandas(gcs_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                        gcs_bucket_name=gcs_bucket_name,
                                        gcs_directory=gcs_directory,
                                        file_name="descriptions_2025Apr28.csv",
                                        exact=False)
df_desc.head()


,Unnamed: 0,file_name,file_type,num_cols,cols,description
0,0,al2023.csv,csv,38,Unique identification number of the institutio...,CSV data file name: al2023.csv. \nOverview des...
1,1,c2023dep.csv,csv,41,Unique identification number of the institutio...,CSV data file name: c2023dep.csv. \nOverview d...
2,2,c2023_a.csv,csv,34,Unique identification number of the institutio...,CSV data file name: c2023_a.csv. \nOverview de...
3,3,c2023_b.csv,csv,41,Unique identification number of the institutio...,CSV data file name: c2023_b.csv. \nOverview de...
4,4,c2023_c.csv,csv,19,Unique identification number of the institutio...,CSV data file name: c2023_c.csv. \nOverview de...


## Read the IPEDS tables from Google Cloud and save in dataframes

In [4]:
dfs = []
for idx in df_desc.index:

    # Read this data from GCP
    df = gct.read_csv_file_into_pandas(gcs_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                       gcs_bucket_name=gcs_bucket_name,
                                       gcs_directory=gcs_directory,
                                       file_name=df_desc.loc[idx, "file_name"],
                                       exact=False)

    # Drop this unneeded column
    if "Unnamed: 0" in df.columns:
        df.drop(columns=["Unnamed: 0"], inplace=True)

        # Get short column names
        dd_loc = df_desc.loc[idx, "description"].find("Data dictionary:")
        data_disc_txt = df_desc.loc[idx, "description"][dd_loc + len("Data dictionary:"):]

        # Split the data dictionary into individual column entries
        dict_elements = data_disc_txt.split(":::,")
        dict_elements = [c.replace(":::.", "") for c in data_disc_txt.split(":::,")]

        # Create a dictionary
        new_colname_dict = {}
        for col_element in dict_elements:
            c_loc = col_element.find("::")
            sn = col_element[:c_loc].strip()
            ln = col_element[c_loc + 2:].strip()
            new_colname_dict[ln] = sn.lower()

        # Add college name
        new_colname_dict["College Name"] = "college_name"

        # replace long column names with short names
        df.rename(columns=new_colname_dict, inplace=True)

    # Add this to list
    dfs.append(dict(table_name=df_desc.loc[idx, "file_name"],
                    df=df))



## Save the dataframes to BigQuery

In [7]:
# Create a bigquery client
client = bigquery.Client(project=os.environ["GOOGLE_CLOUD_PROJECT"])

# Create a dataset if one doesn't already exist
dataset_id = "ipeds"
gct.create_dataset_if_not_exists(project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                 dataset_name=dataset_id)

# Load tables
for df_dict in dfs:

    # Load table
     gct.load_pandas_to_bigquery(project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                 df=df_dict["df"],
                                 dataset_id="{}.{}".format(os.environ["GOOGLE_CLOUD_PROJECT"],
                                                           dataset_id),
                                 table_name=df_dict["table_name"].replace(".csv", ""),
                                 )




Dataset eternal-bongo-435614-b9.ipeds already exists


100%|██████████| 1/1 [00:00<00:00, 33554.43it/s]


GenericGBQException: Reason: 400 POST https://bigquery.googleapis.com/bigquery/v2/projects/eternal-bongo-435614-b9/datasets/ipeds/tables?prettyPrint=false: Invalid field name "Level of student.1". Fields must contain the allowed characters, and be at most 300 characters long. For allowed characters, please refer to https://cloud.google.com/bigquery/docs/schemas#column_names